## Milestone 1 — EDA

Exploratory data analysis for the appliances category of the Amazon Reviews 2023 dataset

In [2]:
import pandas as pd
import duckdb
import requests
from tqdm import tqdm
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import re
import string
from pathlib import Path
import sys
sys.path.append("../src") 

In [ ]:
#adapted from https://github.ubc.ca/mds-2025-26/DSCI_575_adv-mach-learn_students/blob/master/project/milestone1/project_eda_duckdb.ipynb

DATA_DIR = Path("..") / "data"        
RAW_DIR = DATA_DIR / "raw"
CATEGORY = "Appliances"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEWS_URL = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"
META_URL    = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"
REVIEWS_FILE = RAW_DIR / f"{CATEGORY}.jsonl.gz"
META_FILE    = RAW_DIR / f"meta_{CATEGORY}.jsonl.gz"
REVIEWS_PARQUET = RAW_DIR / "reviews_raw.parquet"
META_PARQUET    = RAW_DIR / "meta_raw.parquet"
MERGED_PARQUET  = RAW_DIR / "merged.parquet"

print(REVIEWS_URL)
print(META_URL)


https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Appliances.jsonl.gz
https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Appliances.jsonl.gz


In [4]:

c2 = duckdb.connect()

In [5]:

head_reviews = c2.execute(f"SELECT * FROM read_json_auto('{REVIEWS_URL}') LIMIT 5").df()
head_reviews

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Work great,work great. use a new one every month,[],B01N0TQ0OH,B01N0TQ0OH,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1519317108692,0,True
1,5.0,excellent product,Little on the thin side,[],B07DD2DMXB,B07DD37QPZ,AHWWLSPCJMALVHDDVSUGICL6RUCA,1664746863446,0,True
2,5.0,Happy customer!,"Quick delivery, fixed the issue!",[],B082W3Z9YK,B082W3Z9YK,AHZIJGKEWRTAEOZ673G5B3SNXEGQ,1607225435363,0,True
3,5.0,Amazing value,I wasn't sure whether these were worth it or n...,[],B078W2BJY8,B078W2BJY8,AFGUPTDFAWOHHL4LZDV27ERDNOYQ,1534104184306,0,True
4,5.0,Dryer parts,Easy to install got the product expected to re...,[],B08C9LPCQV,B08C9LPCQV,AELFJFAXQERUSMTXJQ6SYFFRDWMA,1620176603754,0,True


In [7]:
head_meta = c2.execute(f"SELECT * FROM read_json_auto('{META_URL}') LIMIT 5").df()
head_meta

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Industrial & Scientific,"ROVSUN Ice Maker Machine Countertop, Make 44lb...",3.7,61,[【Quick Ice Making】This countertop ice machine...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Our Point of View on the Euhomy Ic...,ROVSUN,"[Appliances, Refrigerators, Freezers & Ice Mak...","{'Brand': '""ROVSUN""', 'Model Name': '""ICM-2005...",B08Z743RRD,None
1,Tools & Home Improvement,"HANSGO Egg Holder for Refrigerator, Deviled Eg...",4.2,75,"[Plastic, Practical Kitchen Storage - Our egg ...",[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': '10 Eggs Egg Holder for Refrigerato...,HANSGO,"[Appliances, Parts & Accessories, Refrigerator...","{'Manufacturer': '""HANSGO""', 'Part Number': '""...",B097BQDGHJ,None
2,Tools & Home Improvement,"Clothes Dryer Drum Slide, General Electric, Ho...",3.5,18,[],"[Brand new dryer drum slide, replaces General ...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],GE,"[Appliances, Parts & Accessories]","{'Manufacturer': '""RPI""', 'Part Number': '""WE1...",B00IN9AGAE,None
3,Tools & Home Improvement,154567702 Dishwasher Lower Wash Arm Assembly f...,4.5,26,[MODEL NUMBER:154567702 Dishwasher Lower Wash ...,[MODEL NUMBER:154567702 Dishwasher Lower Wash ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],folosem,"[Appliances, Parts & Accessories, Dryer Parts ...","{'Manufacturer': '""folosem""', 'Part Number': '...",B0C7K98JZS,None
4,Tools & Home Improvement,Whirlpool W10918546 Igniter,3.8,12,[This is a Genuine OEM Replacement Part.],[Whirlpool Igniter],25.07,[{'thumb': 'https://m.media-amazon.com/images/...,[],Whirlpool,"[Appliances, Parts & Accessories]","{'Manufacturer': '""Whirlpool""', 'Part Number':...",B07QZHQTVJ,None


In [14]:
c2.execute(f"""
    COPY (SELECT * FROM read_json_auto('{REVIEWS_URL}') LIMIT 20000)
    TO '{REVIEWS_PARQUET}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

c2.execute(f"""
    COPY (SELECT * FROM read_json_auto('{META_URL}') LIMIT 20000)
    TO '{META_PARQUET}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")


In [ ]:
c2.execute(f"""
    COPY (
        SELECT r.*, m.title AS product_title, m.price,
               m.average_rating, m.main_category, m.store
        FROM read_parquet('{REVIEWS_PARQUET}') r
        LEFT JOIN read_parquet('{META_PARQUET}') m USING (parent_asin)
    )
    TO '{MERGED_PARQUET}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")


In [18]:
c2.execute(f"SELECT * FROM read_parquet('{MERGED_PARQUET}')").df()


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,price,average_rating,main_category,store
0,5.0,excellent product,Little on the thin side,[],B07DD2DMXB,B07DD37QPZ,AHWWLSPCJMALVHDDVSUGICL6RUCA,1664746863446,0,True,Essential Values 18 Pack Compatible Replacemen...,22.99,4.4,Tools & Home Improvement,Essential Values
1,5.0,Happy customer!,"Quick delivery, fixed the issue!",[],B082W3Z9YK,B082W3Z9YK,AHZIJGKEWRTAEOZ673G5B3SNXEGQ,1607225435363,0,True,279838 Dryer Heating Element by Romalon with R...,NaN,4.5,Tools & Home Improvement,Romalon
2,5.0,DO NOT purchase this ice machine.,After buying this ice machine just 15 months a...,[],B08D6RFV6D,B099ZKQJHK,AEUH4EH6XHROLT7UZPUYU2YKTYMA,1663078878875,0,True,COOLLIFE Compact Countertop Ice Maker Machine ...,NaN,4.1,Industrial & Scientific,COOLLIFE
3,2.0,They don't fit properly,Not the best quality,[],B001TH7GZA,B001TH7H0O,AHCV2CNCOCG6WECDROOUYPDZIFEQ,1610219023865,0,True,"Stanco 5557 Drip Bowl Universal, Porcelain coa...",NaN,4.3,Tools & Home Improvement,Stanco
4,5.0,so far so good,but i havent had it long a year down the road...,[],B09B21HWFM,B09W5PMK5X,AHGAOIZVODNHYMNCBV4DECZH42UQ,1650153667807,2,True,COMFEE’ Washing Machine 2.4 Cu.ft LED Portable...,399.00,3.5,Appliances,COMFEE'
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,5.0,Natural Product,We love the Natural Filters for our Chemex Pou...,[],B017OFOP68,B017OFOP68,AG3TLL5HJQ7FPK3KJCIF4VKL7TYA,1613496241659,0,True,None,NaN,NaN,None,None
19996,3.0,Three Stars,"Refrigerator doesn't recognize this filter, so...",[],B01I3M7K3Q,B01I3M7K3Q,AHR6M4DYJMUKR6T4VB6MFN67MEOA,1486870837000,1,True,None,NaN,NaN,None,None
19997,5.0,Five Stars,As described.,[],B002YJ3YRQ,B002YJ3YRQ,AHR6M4DYJMUKR6T4VB6MFN67MEOA,1477075770000,0,True,None,NaN,NaN,None,None
19998,5.0,Fits most makers,Used to filter a Mr. Coffee. Very good.,[],B00GPXTA0C,B00GPXTA0C,AHVZWMOITJQRTH4BVOCJXWLTFUXQ,1651336382350,0,True,None,NaN,NaN,None,None


In [19]:
c2.execute(f"""
    SELECT
        rating,
        COUNT(*) AS cnt,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM read_parquet('{MERGED_PARQUET}')
    GROUP BY 1
    ORDER BY 1
""").df()


,rating,cnt,pct
0,1.0,1484,7.42
1,2.0,624,3.12
2,3.0,1139,5.70
3,4.0,2304,11.52
4,5.0,14449,72.25


In [22]:
SAMPLE_PER_STRATUM = 50
MIN_TEXT_LEN       = 20
SHORT_MAX          = 100
MEDIUM_MAX         = 500

OUTPUT = RAW_DIR / "stratified_sample.parquet"


In [23]:
c2.execute(f"""
COPY (
WITH labelled AS (
    SELECT
        text, rating, verified_purchase, helpful_vote,
        parent_asin, user_id, timestamp,
        product_title, price,
        average_rating, main_category,

        CASE
            WHEN average_rating >= 4.6 THEN '4.6-5.0'
            WHEN average_rating >= 4.4 THEN '4.4-4.5'
            WHEN average_rating >= 4.1 THEN '4.1-4.3'
            WHEN average_rating >= 3.7 THEN '3.7-4.0'
            WHEN average_rating >= 3.1 THEN '3.1-3.6'
            ELSE                              '<=3.0'
        END AS rating_bucket,

        CASE
            WHEN LENGTH(text) < {SHORT_MAX}  THEN 'short'
            WHEN LENGTH(text) < {MEDIUM_MAX} THEN 'medium'
            ELSE                                    'long'
        END AS len_tier

    FROM read_parquet('{MERGED_PARQUET}')
    WHERE text IS NOT NULL
      AND LENGTH(text) >= {MIN_TEXT_LEN}
      AND average_rating IS NOT NULL
),

one_per_product AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY rating_bucket, len_tier, verified_purchase, parent_asin
            ORDER BY helpful_vote DESC, random()
        ) AS product_rank
    FROM labelled
),

ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY rating_bucket, len_tier, verified_purchase
            ORDER BY helpful_vote DESC, random()
        ) AS stratum_rank
    FROM one_per_product
    WHERE product_rank = 1
)

SELECT * EXCLUDE (product_rank, stratum_rank)
FROM ranked
WHERE stratum_rank <= {SAMPLE_PER_STRATUM}
)
TO '{OUTPUT}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")


In [24]:
df = c2.execute(f"SELECT * FROM read_parquet('{OUTPUT}')").df()
df.head()


,text,rating,verified_purchase,helpful_vote,parent_asin,user_id,timestamp,product_title,price,average_rating,main_category,rating_bucket,len_tier
0,junk parts. I'd have given this one star but n...,2.0,True,3,B07KGT9FZH,AFNYVNFKSEVE73JTPJU2B7TQJPBA,1577483496349,WEN Handyman Q-W0014 Washing Machine Suspensio...,27.08,3.8,Tools & Home Improvement,3.7-4.0,short
1,Waste money use your Alluminum foil is beter a...,1.0,True,3,B09SLP6PDV,AEBKNOGHV5CS4QCTB2H2U2ZCCNPA,1549305167376,"MSDADA Stove Covers, 8 Pcs 0.2 mm 10.6"" x 10.6...",10.99,3.9,Tools & Home Improvement,3.7-4.0,short
2,The wheels aren't wide enough. It won't stay ...,1.0,True,2,B00LGUGJWQ,AEFRT3XFOEKLUN25EETPFZEZJY5Q,1531746501339,GE Appliances WD28X10284 Dishwasher Lower Dish...,163.23,3.9,Tools & Home Improvement,3.7-4.0,short
3,Dose not fit - bad cut cheap material,1.0,True,2,B07WLDGXM8,AEFM6T6D5LRNOC7Z6AFLC2RA3G4Q,1588703814100,Stove Protector Liners Compatible with Samsung...,39.95,3.9,Tools & Home Improvement,3.7-4.0,short
4,These are great for extending your drain hose ...,5.0,True,1,B001E0KLWC,AG4XHL6F47UHLD7Q6W444U3SZYBQ,1574387181914,GE Part Number WH49X301 DR HOSE EXT,39.00,3.7,Industrial & Scientific,3.7-4.0,short


In [25]:
# looking at shape and missingness
df.shape, df.isna().mean().sort_values(ascending=False).head(10)


((1149, 13),
 price                0.304613
 rating               0.000000
 verified_purchase    0.000000
 helpful_vote         0.000000
 text                 0.000000
 parent_asin          0.000000
 user_id              0.000000
 timestamp            0.000000
 product_title        0.000000
 average_rating       0.000000
 dtype: float64)

In [ ]:
# top product categories from metadata
c2.execute(f"""
    SELECT
        main_category,
        COUNT(*) AS cnt
    FROM read_parquet('{OUTPUT}')
    WHERE main_category IS NOT NULL
    GROUP BY 1
    ORDER BY cnt DESC
    LIMIT 10
""").df()


,main_category,cnt
0,Amazon Home,390
1,Appliances,342
2,Tools & Home Improvement,329
3,Industrial & Scientific,54
4,Grocery,6
5,Health & Personal Care,5
6,Baby,5
7,Sports & Outdoors,4
8,Cell Phones & Accessories,2
9,AMAZON FASHION,2


In [ ]:
# rating vs price in buckets
c2.execute(f"""
    SELECT
        CASE
            WHEN price < 10 THEN '<10'
            WHEN price < 25 THEN '10-24'
            WHEN price < 50 THEN '25-49'
            WHEN price < 100 THEN '50-99'
            ELSE '100+'
        END AS price_bucket,
        ROUND(AVG(rating), 3) AS avg_rating,
        COUNT(*) AS n
    FROM read_parquet('{OUTPUT}')
    WHERE price IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").df()


,price_bucket,avg_rating,n
0,10-24,4.173,284
1,100+,3.982,168
2,25-49,3.944,160
3,50-99,3.822,73
4,<10,3.789,114


In [ ]:
#an overview of the dataset (fields, size, example records)

overview = pd.DataFrame({
    "field": df.columns,
    "dtype": df.dtypes.astype(str)
})
overview

,field,dtype
text,text,object
rating,rating,float64
verified_purchase,verified_purchase,bool
helpful_vote,helpful_vote,int64
parent_asin,parent_asin,object
user_id,user_id,object
timestamp,timestamp,int64
product_title,product_title,object
price,price,float64
average_rating,average_rating,float64


In [40]:
#inspection of sample records
c2.execute(f"""
    SELECT *
    FROM read_parquet('{OUTPUT}')
    LIMIT 10
""").df()

,text,rating,verified_purchase,helpful_vote,parent_asin,user_id,timestamp,product_title,price,average_rating,main_category,rating_bucket,len_tier
0,junk parts. I'd have given this one star but n...,2.0,True,3,B07KGT9FZH,AFNYVNFKSEVE73JTPJU2B7TQJPBA,1577483496349,WEN Handyman Q-W0014 Washing Machine Suspensio...,27.08,3.8,Tools & Home Improvement,3.7-4.0,short
1,Waste money use your Alluminum foil is beter a...,1.0,True,3,B09SLP6PDV,AEBKNOGHV5CS4QCTB2H2U2ZCCNPA,1549305167376,"MSDADA Stove Covers, 8 Pcs 0.2 mm 10.6"" x 10.6...",10.99,3.9,Tools & Home Improvement,3.7-4.0,short
2,The wheels aren't wide enough. It won't stay ...,1.0,True,2,B00LGUGJWQ,AEFRT3XFOEKLUN25EETPFZEZJY5Q,1531746501339,GE Appliances WD28X10284 Dishwasher Lower Dish...,163.23,3.9,Tools & Home Improvement,3.7-4.0,short
3,Dose not fit - bad cut cheap material,1.0,True,2,B07WLDGXM8,AEFM6T6D5LRNOC7Z6AFLC2RA3G4Q,1588703814100,Stove Protector Liners Compatible with Samsung...,39.95,3.9,Tools & Home Improvement,3.7-4.0,short
4,These are great for extending your drain hose ...,5.0,True,1,B001E0KLWC,AG4XHL6F47UHLD7Q6W444U3SZYBQ,1574387181914,GE Part Number WH49X301 DR HOSE EXT,39.00,3.7,Industrial & Scientific,3.7-4.0,short
5,Great product for the price!!!!!,4.0,True,1,B005HK6OP8,AHJ5XFP37YL2CWI5HGGJPADXJOUA,1481334495000,"Golden Vantage 36"" Stainless Steel Island Moun...",NaN,3.8,Appliances,3.7-4.0,short
6,Love this set of gas burner covers for my old ...,5.0,True,1,B06XP8WLFS,AE7WXPEQCOBLWYM5LFA36I3XNJZQ,1485222649000,Reston Lloyd G-105-B Square Gas Stove Burner C...,16.97,3.8,Amazon Home,3.7-4.0,short
7,Best Ice Maker ever. It works perfect every time.,5.0,True,1,B07M8JMYHQ,AHPKWLDWSOKV5FTJLF2L6SYHKVEQ,1508369547615,Luma Comfort Clear Ice Cube Maker Machine | Fi...,249.99,3.9,Appliances,3.7-4.0,short
8,"Installed in my dishwasher and ran it once, it...",1.0,True,1,B09T794K7S,AFKEZLRPEKN3CJORVM4UN4FEUKOQ,1667138590356,W10518394 Dishwasher Heating Element for Amana...,30.70,3.9,Tools & Home Improvement,3.7-4.0,short
9,Does the job - card is cooler now.,5.0,True,1,B00SY1W96K,AHTDWYY75GI6PZZGZ7MMT7RZVAFA,1418429171000,EVGA GTX 960 Backplate 100-BP-2968-B9,NaN,4.0,Computers,3.7-4.0,short


selection and justification of fields for retrieval

- `text`: core review content; primary retrieval target.  
- `product_title`: appended to review text to add product context.  
- `rating`: quick sentiment signal and qualitative comparison.  
- `parent_asin`: product-level grouping; helps de-duplicate and link metadata.  
- `verified_purchase`: quality/trust signal for reviews.  
- `helpful_vote`: usefulness signal for ranking interpretation.  
- `timestamp`: supports temporal analysis if needed.  
- `average_rating`, `price`, `main_category`, `store`: metadata context for interpretation and analysis.


description of text preprocessing decisions

- **BM25 (`src/bm25.py`)**: simple tokenizer that **lowercases and splits on whitespace**; no stopword removal or punctuation stripping.  
- Review `text` is concatenated with `product_title` into a single `chunk_text`.  
- **Semantic (`src/semantic.py`)**: uses the same `chunk_text` **without additional cleaning**; embeddings are built from raw text + title.  
- Reviews with missing text are still handled safely (converted to empty string before concatenation).  
